In [ ]:

# ============================================================
# HBCD HYPOTHESIS 1
# SIMULATION-BASED POWER ANALYSIS WITH FIT DIAGNOSTICS
#
# 20 states | 1,500 simulated infants
# 3 moderators | 10 effect sizes | 1,000 simulations
# Total: 30,000 simulated datasets
#
# L-BFGS first, Powell fallback
# Text-based tqdm progress bar
# Separate failure categories and optimizer diagnostics
# ============================================================

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import truncnorm
from statsmodels.regression.mixed_linear_model import MixedLM
from tqdm import tqdm

# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 2026

N_TOTAL = 1915
N_HIPPO = 1552
N_ANALYSIS = 1500

N_STATES = 20
N_SITES = 27

N_SIM = 1000
ALPHA = 0.05
TARGET_POWER = 0.95

EFFECTS = [
    0.00, 0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40, 0.50
]

MODERATORS = [
    "cash_benefits",
    "rpp",
    "medicaid"
]

OPTIMIZERS = ["lbfgs", "powell"]

OUTPUT_FILE = "hbcd_h1_power_diagnostics_results.csv"
FIT_LOG_FILE = "hbcd_h1_individual_fit_diagnostics.csv"

# Assumed variance components
VAR_STATE = 0.03
VAR_SITE = 0.04
VAR_FAMILY = 0.03
VAR_RESIDUAL = 0.90

# Assumed fixed-effect coefficients
BETA_INCOME = 0.10
BETA_MODERATOR = 0.05
BETA_GESTATIONAL = 0.08
BETA_SCAN_AGE = 0.10
BETA_SEX = 0.05
BETA_ICV = 0.25

# ============================================================
# 2. STATE-LEVEL MODERATORS
# ============================================================

STATE_ROWS = [
    ("Alabama", 255, 0.89, 0),
    ("Arkansas", 249, 0.87, 1),
    ("California", 778, 1.12, 1),
    ("Georgia", 287, 0.96, 0),
    ("Illinois", 465, 0.99, 1),
    ("Maryland", 584, 1.05, 1),
    ("Massachusetts", 583, 1.08, 1),
    ("Minnesota", 542, 0.98, 1),
    ("Missouri", 308, 0.91, 1),
    ("New Mexico", 458, 0.91, 1),
    ("New York", 586, 1.08, 1),
    ("North Carolina", 283, 0.94, 1),
    ("Ohio", 486, 0.92, 1),
    ("Oklahoma", 300, 0.89, 1),
    ("Oregon", 418, 1.05, 1),
    ("Pennsylvania", 349, 0.97, 1),
    ("Tennessee", 341, 0.92, 0),
    ("Vermont", 631, 0.97, 1),
    ("Virginia", 423, 1.01, 1),
    ("Wisconsin", 490, 0.93, 0)
]

state_df = pd.DataFrame(
    STATE_ROWS,
    columns=[
        "state", "cash_benefits", "rpp", "medicaid"
    ]
)

for col in ["cash_benefits", "rpp"]:
    state_df[col + "_z"] = (
        (state_df[col] - state_df[col].mean())
        / state_df[col].std(ddof=0)
    )

# ============================================================
# 3. HBCD DEMOGRAPHIC DISTRIBUTIONS
# ============================================================

RACE_LABELS = [
    "White", "Black", "Asian",
    "AIAN", "Mixed", "Other"
]

RACE_COUNTS = np.array([
    1050, 413, 59, 15, 273, 105
])

EDUCATION_COUNTS = np.array([
    382, 432, 439, 662
])

INCOME_COUNTS = np.array([
    193, 94, 56, 75, 92,
    131, 192, 194, 564, 324
])

INCOME_MIDPOINTS = np.array([
    10000, 12500, 17500, 22500, 30000,
    42500, 62500, 87500, 150000, 200000
], dtype=float)

RACE_PROBS = RACE_COUNTS / RACE_COUNTS.sum()
EDUCATION_PROBS = EDUCATION_COUNTS / EDUCATION_COUNTS.sum()
INCOME_PROBS = INCOME_COUNTS / INCOME_COUNTS.sum()

# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def standardize(x):
    x = np.asarray(x, dtype=float)
    sd = np.std(x, ddof=0)
    if sd == 0:
        return np.zeros_like(x)
    return (x - np.mean(x)) / sd


def truncated_normal(mean, sd, low, high, n, rng):
    a = (low - mean) / sd
    b = (high - mean) / sd

    return truncnorm.rvs(
        a, b,
        loc=mean,
        scale=sd,
        size=n,
        random_state=rng
    )


# ============================================================
# 5. GENERATE SIMULATED SAMPLE
# ============================================================

def generate_design(n, rng):

    state_ids = np.arange(n) % N_STATES
    rng.shuffle(state_ids)

    site_ids = np.empty(n, dtype=int)

    sites_per_state = [
        2 if s < 7 else 1
        for s in range(N_STATES)
    ]

    site_counter = 0

    for s, count in enumerate(sites_per_state):

        idx = np.where(state_ids == s)[0]

        available_sites = np.arange(
            site_counter,
            site_counter + count
        )

        site_ids[idx] = rng.choice(
            available_sites,
            size=len(idx)
        )

        site_counter += count

    family_ids = np.arange(n)

    for s in range(N_STATES):

        idx = np.where(state_ids == s)[0]
        rng.shuffle(idx)

        n_pairs = int(len(idx) * 0.05)

        for j in range(n_pairs):
            a = idx[2 * j]
            b = idx[2 * j + 1]
            family_ids[b] = family_ids[a]

    gestational_age = truncated_normal(
        38.52, 1.79, 25, 45, n, rng
    )

    scan_age = truncated_normal(
        3.05, 1.82, 0, 16, n, rng
    )

    sex_male = rng.binomial(
        1, 1005 / 1915, n
    )

    race = rng.choice(
        RACE_LABELS,
        size=n,
        p=RACE_PROBS
    )

    hispanic = rng.binomial(
        1, 389 / 1915, n
    )

    education = rng.choice(
        np.arange(4),
        size=n,
        p=EDUCATION_PROBS
    )

    income_band = rng.choice(
        np.arange(10),
        size=n,
        p=INCOME_PROBS
    )

    family_income = INCOME_MIDPOINTS[income_band]

    income_z = standardize(
        np.log(family_income)
    )

    icv_z = standardize(
        rng.normal(size=n)
    )

    manufacturer_by_site = rng.choice(
        ["GE", "Philips", "Siemens"],
        size=N_SITES,
        p=[0.25, 0.15, 0.60]
    )

    df = pd.DataFrame({
        "state_id": state_ids,
        "site_id": site_ids,
        "family_id": family_ids,
        "gestational_z": standardize(gestational_age),
        "scan_age_z": standardize(scan_age),
        "sex_male": sex_male,
        "race": race,
        "hispanic": hispanic,
        "education": education,
        "income_z": income_z,
        "icv_z": icv_z,
        "manufacturer": manufacturer_by_site[site_ids]
    })

    df["state"] = state_df.loc[
        df["state_id"], "state"
    ].to_numpy()

    df = df.merge(
        state_df[
            [
                "state",
                "cash_benefits_z",
                "rpp_z",
                "medicaid"
            ]
        ],
        on="state",
        how="left",
        validate="many_to_one"
    )

    return df


# ============================================================
# 6. SIMULATE HIPPOCAMPAL VOLUME
# ============================================================

def simulate_outcome(df, moderator, beta, rng):

    n = len(df)

    moderator_col = {
        "cash_benefits": "cash_benefits_z",
        "rpp": "rpp_z",
        "medicaid": "medicaid"
    }[moderator]

    m = df[moderator_col].to_numpy()
    x = df["income_z"].to_numpy()

    state_effects = rng.normal(
        0, np.sqrt(VAR_STATE), N_STATES
    )

    site_effects = rng.normal(
        0, np.sqrt(VAR_SITE), N_SITES
    )

    family_codes, _ = pd.factorize(
        df["family_id"]
    )

    family_effects = rng.normal(
        0,
        np.sqrt(VAR_FAMILY),
        len(np.unique(family_codes))
    )

    race_effect = {
        "White": 0.0,
        "Black": 0.02,
        "Asian": -0.02,
        "AIAN": 0.01,
        "Mixed": 0.01,
        "Other": 0.0
    }

    education_effect = np.array([
        -0.04, -0.02, 0.00, 0.02
    ])

    manufacturer_effect = {
        "GE": 0.02,
        "Philips": -0.02,
        "Siemens": 0.0
    }

    fixed = (
        BETA_INCOME * x
        + BETA_MODERATOR * m
        + beta * x * m
        + BETA_GESTATIONAL * df["gestational_z"].to_numpy()
        + BETA_SCAN_AGE * df["scan_age_z"].to_numpy()
        + BETA_SEX * df["sex_male"].to_numpy()
        + BETA_ICV * df["icv_z"].to_numpy()
        + df["race"].map(race_effect).to_numpy()
        + education_effect[df["education"].to_numpy()]
        + 0.02 * df["hispanic"].to_numpy()
        + df["manufacturer"].map(
            manufacturer_effect
        ).to_numpy()
    )

    random_part = (
        state_effects[df["state_id"].to_numpy()]
        + site_effects[df["site_id"].to_numpy()]
        + family_effects[family_codes]
        + rng.normal(
            0, np.sqrt(VAR_RESIDUAL), n
        )
    )

    return 1220 + 187 * (fixed + random_part)


# ============================================================
# 7. FIT MODEL WITH OPTIMIZER FALLBACK
# ============================================================

def fit_interaction(df, moderator):

    moderator_col = {
        "cash_benefits": "cash_benefits_z",
        "rpp": "rpp_z",
        "medicaid": "medicaid"
    }[moderator]

    data = df.copy()

    data["moderator_value"] = data[moderator_col]

    data["interaction"] = (
        data["income_z"] * data["moderator_value"]
    )

    data["hippo_z"] = (
        data["hippocampal_volume"] - 1220
    ) / 187

    formula = (
        "hippo_z ~ income_z + moderator_value "
        "+ interaction + gestational_z + scan_age_z "
        "+ sex_male + icv_z + hispanic "
        "+ C(race) + C(education) + C(manufacturer)"
    )

    attempts = []

    try:
        model = MixedLM.from_formula(
            formula,
            data=data,
            groups="state_id",
            re_formula="1",
            vc_formula={
                "site": "0 + C(site_id)",
                "family": "0 + C(family_id)"
            }
        )

    except Exception as e:
        return {
            "status": "exception",
            "optimizer": "none",
            "pvalue": np.nan,
            "attempts": [],
            "error": type(e).__name__ + ": " + str(e)
        }

    for optimizer in OPTIMIZERS:

        start = time.perf_counter()

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                fit = model.fit(
                    reml=False,
                    method=optimizer,
                    maxiter=500,
                    disp=False
                )

            elapsed = time.perf_counter() - start

            warning_messages = " | ".join(
                str(w.message) for w in caught
            )

            if not fit.converged:

                attempts.append({
                    "optimizer": optimizer,
                    "status": "nonconvergence",
                    "seconds": elapsed,
                    "warnings": warning_messages
                })

                continue

            p = float(fit.pvalues["interaction"])

            if not np.isfinite(p):

                attempts.append({
                    "optimizer": optimizer,
                    "status": "invalid_pvalue",
                    "seconds": elapsed,
                    "warnings": warning_messages
                })

                continue

            # Variance components
            variance_lookup = dict(
                zip(model.exog_vc.names, fit.vcomp)
            )

            state_var = float(
                fit.cov_re.iloc[0, 0]
            )

            site_var = float(
                variance_lookup.get("site", np.nan)
            )

            family_var = float(
                variance_lookup.get("family", np.nan)
            )

            residual_var = float(fit.scale)

            variances = np.array([
                state_var,
                site_var,
                family_var
            ])

            boundary = bool(
                np.any(
                    variances
                    < 1e-4 * residual_var
                )
            )

            attempts.append({
                "optimizer": optimizer,
                "status": "success",
                "seconds": elapsed,
                "warnings": warning_messages
            })

            return {
                "status": "success",
                "optimizer": optimizer,
                "pvalue": p,
                "beta_estimate": float(
                    fit.params["interaction"]
                ),
                "se_estimate": float(
                    fit.bse["interaction"]
                ),
                "log_likelihood": float(fit.llf),
                "state_variance": state_var,
                "site_variance": site_var,
                "family_variance": family_var,
                "residual_variance": residual_var,
                "boundary_component": boundary,
                "attempts": attempts,
                "error": ""
            }

        except Exception as e:

            elapsed = time.perf_counter() - start

            attempts.append({
                "optimizer": optimizer,
                "status": "exception",
                "seconds": elapsed,
                "warnings": "",
                "error": type(e).__name__ + ": " + str(e)
            })

    # If all optimizers fail, classify by final attempt.
    final_status = (
        attempts[-1]["status"]
        if attempts else "exception"
    )

    return {
        "status": final_status,
        "optimizer": "none",
        "pvalue": np.nan,
        "attempts": attempts,
        "error": (
            attempts[-1].get("error", "")
            if attempts else ""
        )
    }


# ============================================================
# 8. RUN MONTE CARLO SIMULATIONS
# ============================================================

def run_power_analysis():

    rng = np.random.default_rng(SEED)

    total = len(MODERATORS) * len(EFFECTS) * N_SIM

    summary_records = []
    fit_records = []

    with tqdm(
        total=total,
        desc="HBCD H1 simulations",
        unit="fit",
        ncols=100,
        mininterval=1
    ) as progress:

        for moderator in MODERATORS:

            for beta in EFFECTS:

                pvalues = []

                counts = {
                    "success_lbfgs": 0,
                    "success_powell": 0,
                    "nonconvergence": 0,
                    "invalid_pvalue": 0,
                    "exception": 0
                }

                for sim in range(N_SIM):

                    local_rng = np.random.default_rng(
                        rng.integers(0, 2**32 - 1)
                    )

                    try:

                        df = generate_design(
                            N_ANALYSIS,
                            local_rng
                        )

                        df["hippocampal_volume"] = (
                            simulate_outcome(
                                df,
                                moderator,
                                beta,
                                local_rng
                            )
                        )

                        result = fit_interaction(
                            df, moderator
                        )

                    except Exception as e:

                        result = {
                            "status": "exception",
                            "optimizer": "none",
                            "pvalue": np.nan,
                            "attempts": [],
                            "error": (
                                type(e).__name__
                                + ": " + str(e)
                            )
                        }

                    status = result["status"]

                    if status == "success":

                        pvalues.append(result["pvalue"])

                        counts[
                            "success_" + result["optimizer"]
                        ] += 1

                    else:
                        counts[status] += 1

                    # Record each optimizer attempt separately.
                    for attempt_number, attempt in enumerate(
                        result.get("attempts", []), start=1
                    ):

                        fit_records.append({
                            "moderator": moderator,
                            "interaction_beta": beta,
                            "simulation": sim + 1,
                            "attempt_number": attempt_number,
                            "optimizer": attempt["optimizer"],
                            "attempt_status": attempt["status"],
                            "seconds": attempt["seconds"],
                            "warnings": attempt.get(
                                "warnings", ""
                            ),
                            "error": attempt.get(
                                "error", ""
                            )
                        })

                    # Record final fit outcome separately.
                    fit_records.append({
                        "moderator": moderator,
                        "interaction_beta": beta,
                        "simulation": sim + 1,
                        "attempt_number": 0,
                        "optimizer": result["optimizer"],
                        "attempt_status": (
                            "final_" + status
                        ),
                        "seconds": np.nan,
                        "warnings": "",
                        "error": result.get("error", ""),
                        "pvalue": result.get(
                            "pvalue", np.nan
                        ),
                        "beta_estimate": result.get(
                            "beta_estimate", np.nan
                        ),
                        "se_estimate": result.get(
                            "se_estimate", np.nan
                        ),
                        "log_likelihood": result.get(
                            "log_likelihood", np.nan
                        ),
                        "state_variance": result.get(
                            "state_variance", np.nan
                        ),
                        "site_variance": result.get(
                            "site_variance", np.nan
                        ),
                        "family_variance": result.get(
                            "family_variance", np.nan
                        ),
                        "residual_variance": result.get(
                            "residual_variance", np.nan
                        ),
                        "boundary_component": result.get(
                            "boundary_component", np.nan
                        )
                    })

                    progress.update(1)

                n_valid = len(pvalues)
                n_failed = N_SIM - n_valid

                if n_valid > 0:

                    power = np.mean(
                        np.asarray(pvalues) < ALPHA
                    )

                    mc_se = np.sqrt(
                        power * (1 - power) / n_valid
                    )

                else:
                    power = np.nan
                    mc_se = np.nan

                summary_records.append({
                    "moderator": moderator,
                    "interaction_beta": beta,
                    "n_analysis": N_ANALYSIS,
                    "n_sim": N_SIM,
                    "n_valid": n_valid,
                    "n_failed": n_failed,
                    "convergence_rate": n_valid / N_SIM,
                    "power_unadjusted": power,
                    "mc_se": mc_se,
                    **counts
                })

                # Save completed scenarios.
                pd.DataFrame(summary_records).to_csv(
                    OUTPUT_FILE,
                    index=False
                )

                pd.DataFrame(fit_records).to_csv(
                    FIT_LOG_FILE,
                    index=False
                )

                progress.set_postfix({
                    "mod": moderator,
                    "beta": beta,
                    "valid": n_valid,
                    "power": (
                        round(power, 3)
                        if np.isfinite(power)
                        else "NA"
                    )
                })

    return (
        pd.DataFrame(summary_records),
        pd.DataFrame(fit_records)
    )


# ============================================================
# 9. EXECUTE
# ============================================================

results, fit_diagnostics = run_power_analysis()

print("\nSIMULATION COMPLETE")

display(results)

# ============================================================
# 10. CONVERGENCE SUMMARY
# ============================================================

convergence_summary = (
    results.groupby("moderator")
    .agg(
        mean_convergence=("convergence_rate", "mean"),
        total_valid=("n_valid", "sum"),
        total_failed=("n_failed", "sum"),
        lbfgs_success=("success_lbfgs", "sum"),
        powell_success=("success_powell", "sum"),
        nonconvergence=("nonconvergence", "sum"),
        invalid_pvalue=("invalid_pvalue", "sum"),
        exception=("exception", "sum")
    )
    .reset_index()
)

print("\nCONVERGENCE SUMMARY")
display(convergence_summary.round(4))

# ============================================================
# 11. TYPE I ERROR CALIBRATION
# ============================================================

type1_error = results[
    results["interaction_beta"] == 0
][[
    "moderator",
    "power_unadjusted",
    "mc_se",
    "convergence_rate",
    "n_failed"
]].copy()

type1_error = type1_error.rename(
    columns={
        "power_unadjusted": "type1_error_rate"
    }
)

print("\nTYPE I ERROR CALIBRATION")
display(type1_error.round(4))

# ============================================================
# 12. MINIMUM TESTED EFFECT FOR 95% POWER
# ============================================================

minimum_effects = []

for moderator in MODERATORS:

    sub = results[
        (results["moderator"] == moderator)
        & (results["interaction_beta"] > 0)
        & (results["power_unadjusted"] >= TARGET_POWER)
    ].sort_values("interaction_beta")

    if len(sub):

        first = sub.iloc[0]

        minimum_effects.append({
            "moderator": moderator,
            "minimum_tested_beta": first["interaction_beta"],
            "estimated_power": first["power_unadjusted"],
            "convergence_rate": first["convergence_rate"]
        })

    else:

        minimum_effects.append({
            "moderator": moderator,
            "minimum_tested_beta": np.nan,
            "estimated_power": np.nan,
            "convergence_rate": np.nan
        })

print("\nMINIMUM TESTED EFFECT REACHING 95% POWER")
display(pd.DataFrame(minimum_effects))

# ============================================================
# 13. POWER CURVES
# ============================================================

plt.figure(figsize=(9, 6))

labels = {
    "cash_benefits": "Cash benefits",
    "rpp": "Regional Price Parity",
    "medicaid": "Medicaid expansion"
}

for moderator in MODERATORS:

    sub = results[
        results["moderator"] == moderator
    ].sort_values("interaction_beta")

    plt.plot(
        sub["interaction_beta"],
        sub["power_unadjusted"],
        marker="o",
        linewidth=2,
        label=labels[moderator]
    )

plt.axhline(
    TARGET_POWER,
    linestyle="--",
    color="red",
    label="95% target"
)

plt.xlabel("Assumed interaction coefficient")
plt.ylabel("Estimated power")
plt.title("HBCD H1: Simulation-Based Power Analysis")
plt.ylim(0, 1.02)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# ============================================================
# 14. OPTIMIZER USAGE
# ============================================================

optimizer_usage = (
    results.groupby("moderator")[
        ["success_lbfgs", "success_powell"]
    ]
    .sum()
    .reset_index()
)

print("\nOPTIMIZER USAGE")
display(optimizer_usage)

# ============================================================
# 15. BOUNDARY SOLUTIONS
# ============================================================

final_fits = fit_diagnostics[
    fit_diagnostics["attempt_status"] == "final_success"
].copy()

print("\nBOUNDARY SOLUTION RATES")

if len(final_fits):

    boundary_summary = (
        final_fits.groupby("optimizer")
        ["boundary_component"]
        .agg(["count", "mean"])
        .reset_index()
    )

    display(boundary_summary)

# ============================================================
# 16. FINAL OUTPUT
# ============================================================

print("\nFiles saved:")
print(OUTPUT_FILE)
print(FIT_LOG_FILE)

print("\nTotal simulated datasets:", len(MODERATORS) * len(EFFECTS) * N_SIM)
print("Target power:", TARGET_POWER)
print("Significance threshold:", ALPHA)

print(
    "\nNote: Power is conditional on successful fits "
    "and does not include BH-FDR correction."
)


In [ ]:

import numpy as np
import pandas as pd

# Load individual model results
df = pd.read_csv(
    "hbcd_h1_individual_fit_diagnostics.csv"
)

# Keep only final successful fits
df = df[
    df["attempt_status"] == "final_success"
].copy()

# Three moderator tests in Hypothesis 1
alpha = 0.05
n_tests = 3
bonferroni_alpha = alpha / n_tests

# Rejection indicators
df["significant_unadjusted"] = (
    df["pvalue"] < alpha
)

df["significant_bonferroni"] = (
    df["pvalue"] < bonferroni_alpha
)

# Summarize empirical power
power_comparison = (
    df.groupby(
        ["moderator", "interaction_beta"]
    )
    .agg(
        n_valid=("pvalue", "count"),
        power_unadjusted=(
            "significant_unadjusted", "mean"
        ),
        power_bonferroni=(
            "significant_bonferroni", "mean"
        )
    )
    .reset_index()
)

power_comparison.to_csv(
    "hbcd_h1_power_bonferroni_comparison.csv",
    index=False
)

display(power_comparison.round(4))

# Find smallest tested effect reaching 95% power
minimum_effects = []

for moderator, group in power_comparison.groupby(
    "moderator"
):
    eligible = group[
        (group["interaction_beta"] > 0)
        & (group["power_bonferroni"] >= 0.95)
    ].sort_values("interaction_beta")

    if not eligible.empty:
        first = eligible.iloc[0]
        minimum_effects.append({
            "moderator": moderator,
            "minimum_tested_beta": first["interaction_beta"],
            "power_bonferroni": first["power_bonferroni"]
        })
    else:
        minimum_effects.append({
            "moderator": moderator,
            "minimum_tested_beta": np.nan,
            "power_bonferroni": np.nan
        })

print("Minimum tested effects with >=95% power:")
display(pd.DataFrame(minimum_effects))


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# --------------------------------------------------
# Figure settings
# --------------------------------------------------

plt.rcParams.update({
    "font.family": "Helvetica",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white"
})

# Purple color palette
dark_purple = "#45216F"
medium_purple = "#9063C2"
pink_purple = "#C65B96"
light_purple = "#EAE1F3"
text_purple = "#32174D"
target_color = "#D14C65"

styles = {
    "cash_benefits": {
        "label": "Cash assistance",
        "color": dark_purple,
        "linestyle": "-",
        "marker": "o"
    },
    "rpp": {
        "label": "Regional Price Parity",
        "color": medium_purple,
        "linestyle": "--",
        "marker": "s"
    },
    "medicaid": {
        "label": "Medicaid expansion",
        "color": pink_purple,
        "linestyle": ":",
        "marker": "^"
    }
}

# --------------------------------------------------
# Create figure
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6.5))

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

for moderator, style in styles.items():

    subset = results[
        results["moderator"] == moderator
    ].sort_values("interaction_beta")

    ax.plot(
        subset["interaction_beta"],
        subset["power_unadjusted"],
        label=style["label"],
        color=style["color"],
        linestyle=style["linestyle"],
        marker=style["marker"],
        linewidth=2.8,
        markersize=7,
        markeredgecolor="white",
        markeredgewidth=0.7,
        zorder=3
    )

# 95% power target
ax.axhline(
    y=0.95,
    color=target_color,
    linestyle="--",
    linewidth=2,
    label="95% target",
    zorder=2
)

# --------------------------------------------------
# Labels and formatting
# --------------------------------------------------

ax.set_title(
    "H1: Simulation-Based Power Analysis",
    fontweight="bold",
    color=dark_purple,
    pad=18
)

ax.set_xlabel(
    "Assumed interaction coefficients (β)",
    fontweight="bold",
    color=text_purple,
    labelpad=12
)

ax.set_ylabel(
    "Estimated power",
    fontweight="bold",
    color=text_purple,
    labelpad=12
)

ax.set_xlim(-0.01, 0.51)
ax.set_ylim(0, 1.03)

ax.xaxis.set_major_locator(MultipleLocator(0.05))
ax.yaxis.set_major_locator(MultipleLocator(0.2))

ax.tick_params(
    axis="both",
    colors=text_purple,
    length=5,
    width=1
)

# Subtle horizontal gridlines
ax.grid(
    axis="y",
    color=light_purple,
    linewidth=0.8,
    alpha=0.65
)

ax.grid(axis="x", visible=False)

# Remove upper and right borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for side in ["left", "bottom"]:
    ax.spines[side].set_color(dark_purple)
    ax.spines[side].set_linewidth(1.2)

# --------------------------------------------------
# Legend
# --------------------------------------------------

legend = ax.legend(
    loc="lower right",
    frameon=True,
    fancybox=True,
    facecolor="white",
    edgecolor=light_purple,
    framealpha=0.97,
    borderpad=1,
    handlelength=3
)

for text in legend.get_texts():
    text.set_color(text_purple)

# --------------------------------------------------
# Export
# --------------------------------------------------

plt.tight_layout()

plt.savefig(
    "HBCD_H1_Power_Analysis_Purple.png",
    dpi=1000,
    bbox_inches="tight"
)

plt.savefig(
    "HBCD_H1_Power_Analysis_Purple.pdf",
    bbox_inches="tight"
)

plt.show()
